In [1]:
import os
from pytubefix import YouTube
import argparse
import re
import whisper
import sys
import pandas as pd
import transformers
import tensorflow as tf
from datasets import load_dataset
from datasets import Dataset, DatasetDict
import torch
import contractions
from emoji import demojize
import numpy as np
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, AdamWeightDecay, RobertaTokenizer, AutoModelForSequenceClassification
from transformers import Trainer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt

SLANG_DICT = {
    # Common abbreviations
    "lol": "laughing out loud",
    "omg": "oh my god",
    "btw": "by the way",
    "idk": "i don't know",
    "tbh": "to be honest",
    "imo": "in my opinion",
    "smh": "shaking my head",
    "afaik": "as far as i know",
    "fyi": "for your information",
    "np": "no problem",
    "thx": "thanks",
    "pls": "please",
    "asap": "as soon as possible",
    "jk": "just kidding",
    "nvm": "never mind",
    "brb": "be right back",
    "gtg": "got to go",
    "irl": "in real life",
    "dm": "direct message",
    "tmi": "too much information",
    
    # Emphatic expressions
    "wtf": "what the fuck",
    "omfg": "oh my fucking god",
    "stfu": "shut the fuck up",
    "fml": "fuck my life",
    "rofl": "rolling on the floor laughing",
    "lmao": "laughing my ass off",
    "lmfao": "laughing my fucking ass off",
    
    # Modern internet slang
    "sus": "suspicious",
    "ghosting": "ignoring someone",
    "simp": "someone idolizing others",
    "flex": "showing off",
    "clout": "influence",
    "vibe": "mood",
    "yeet": "throw forcefully",
    "lit": "exciting",
    "salty": "bitter/angry",
    "cap": "lie",
    "no cap": "truth",
    "bet": "agreement",
    "ship": "relationship",
    "stan": "obsessed fan",
    
    # Textspeak conversions
    "u": "you",
    "ur": "your",
    "r": "are",
    "y": "why",
    "k": "okay",
    "ppl": "people",
    "def": "definitely",
    "prob": "probably",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to"
}
SLANG_DICT.update({
    "gg": "good game",
    "op": "overpowered",
    "nerf": "reduce power",
    "pog": "awesome"
})
def optimized_preprocessor(text):
    """
    Minimal yet effective preprocessing for transformer models
    Returns: Cleaned text string
    """
    # Convert emojis to text descriptions
    text = demojize(text, delimiters=(" ", " "))  # 👌🏾 → :ok_hand_medium-dark_skin_tone:
    
    # Expand slang/abbreviations
    text = ' '.join([SLANG_DICT.get(word.lower(), word) for word in text.split()])
    
    # Handle repeated characters (e.g., "loooool" → "lool")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove remaining special characters (keep apostrophes and basic punctuation)
    text = re.sub(r"[^a-zA-Z0-9\s!?,;:'\-]", "", text)

    # Handle contractions (e.g., "can't" → "cannot")
    text = contractions.fix(text)
    
    # Remove user mentions (@username)
    text = re.sub(r"@\w+", "[USER]", text)
    
    # Remove URLs
    text = re.sub(r"http\S+", "[URL]", text)
    
    # Normalize numbers
    text = re.sub(r"\d+", "[NUM]", text)
    
    # Convert to lowercase
    text = text.lower()
    
    return text
def remove_placeholders(text):
    """
    Removes placeholders [NUM], [USER], and [URL] from the text.
    """
    # Remove [NUM], [USER], and [URL]
    text = re.sub(r"\[NUM\]", "", text)
    text = re.sub(r"\[USER\]", "", text)
    text = re.sub(r"\[URL\]", "", text)
    
    # Optionally, strip any extra spaces that might be left after removal
    text = ' '.join(text.split())
    
    return text

def transcribe_audio(mp3_file: str, model_name: str = "base") -> str:
    """
    Transcribes the given MP3 file using the specified Whisper model.

    Args:
        mp3_file (str): Path to the MP3 file.
        model_name (str): Name of the Whisper model to use (tiny, base, small, medium, large).

    Returns:
        str: The full transcription text.
    """
    print(f"Loading Whisper model '{model_name}'...")
    model = whisper.load_model(model_name)
    print(f"Transcribing '{mp3_file}'...")
    result = model.transcribe(mp3_file)
    return result["text"]


def split_into_sentences(text: str) -> list:
    """
    Splits a block of text into sentences using a regular expression.
    This regex splits the text at punctuation marks (., !, or ?) followed by whitespace.

    Args:
        text (str): The text to split.

    Returns:
        list: A list of sentences.
    """
    # The regex splits on punctuation that likely ends a sentence.
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Clean up any extra whitespace or empty strings.
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    return sentences


def save_sentences_to_csv(sentences: list, output_file: str) -> None:
    """
    Saves a list of sentences into a CSV file with one column "Sentence".
    Uses UTF-8 encoding with BOM to properly display Romanian characters.
    
    Args:
        sentences (list): List of sentence strings.
        output_file (str): Path to the output CSV file.
    """
    df = pd.DataFrame(sentences, columns=["Sentence"])
    df.to_csv(output_file, index=False, encoding='utf-8-sig')  # Specify encoding here
    print(f"Transcription saved to '{output_file}'.")


def download_youtube_audio(url, destination="video_to_mp3"):
    try:
        yt = YouTube(url)
        title = yt.title
        sanitized_title = "".join(c for c in title if c.isalnum() or c in " _-").rstrip()
        mp3_filename = os.path.join(destination, sanitized_title + ".mp3")

        # Check if the file already exists
        if os.path.exists(mp3_filename):
            print(f"File already downloaded: {mp3_filename}")
            return mp3_filename

        # If not, download it
        video = yt.streams.filter(only_audio=True).first()
        out_file = video.download(output_path=destination)
        base, ext = os.path.splitext(out_file)
        new_file = base + '.mp3'
        os.rename(out_file, new_file)

        print(f"Download complete: {new_file}")
        return new_file

    except Exception as e:
        print("Failed to download audio:", e)
        return None

def handle_mp3_file(path):
    if os.path.isfile(path) and path.lower().endswith(".mp3"):
        print(f"MP3 file found at: {path}")
        return path
    else:
        print("Invalid file path or not an MP3.")
        return None
    
# Translation function using trained model
def translate_text(text, tokenizer, model):
    tokenized = tokenizer([text], return_tensors='np', padding=True, truncation=True, max_length=128)
    out = model.generate(**tokenized, max_length=128)
    return tokenizer.decode(out[0], skip_special_tokens=True)

c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
!pip install sacremoses

   ---------------------------------------- 0.0/897.5 kB ? eta -:--:--
   ---------------------------------------- 10.2/897.5 kB ? eta -:--:--
   ----------------------- ---------------- 522.2/897.5 kB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 897.5/897.5 kB 9.4 MB/s eta 0:00:00


In [2]:
user_input = input("Enter a YouTube link or path to an .mp3 file:\n").strip()
if user_input.startswith("http://") or user_input.startswith("https://"):
    file_path = download_youtube_audio(user_input)
else:
    file_path = handle_mp3_file(user_input)

if not file_path or not os.path.isfile(file_path):
    print(f"Error: The file '{file_path}' does not exist.")
    exit(1)
print(f"File path: {file_path}")
# Transcribe the audio file.
transcription_text = transcribe_audio(file_path, model_name='large')

# Split the transcription into sentences.

sentences = split_into_sentences(transcription_text)

# Determine output file name
output_file = "audio_transcript.csv"

# Save the sentences to CSV.
save_sentences_to_csv(sentences, output_file)
# Reload trained model for translation
tokenizer_translate = AutoTokenizer.from_pretrained("models/translation/")
model_translate = TFAutoModelForSeq2SeqLM.from_pretrained("models/translation/")

File already downloaded: video_to_mp3\O concurentă a fost luată cu targa de medici  SURVIVOR 2025.mp3
File path: video_to_mp3\O concurentă a fost luată cu targa de medici  SURVIVOR 2025.mp3
Loading Whisper model 'large'...


c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device

Transcribing 'video_to_mp3\O concurentă a fost luată cu targa de medici  SURVIVOR 2025.mp3'...


c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to 'audio_transcript.csv'.


c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at models/translation/.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.


FileNotFoundError: [Errno 2] No such file or directory: 'output_file.csv'

In [5]:
df = pd.read_csv("audio_transcript.csv")

# Apply translation using trained model
df["Translation"] = df["Sentence"].astype(str).apply(
    lambda x: translate_text(x, tokenizer=tokenizer_translate, model=model_translate)
)

# Save the result as an Excel file
df.to_csv("translated_data.csv", index=False,encoding='utf-8-sig')
print(f"Translated sentences saved to 'translated_data.csv'.")



Translated sentences saved to 'translated_data.csv'.
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\IPython\core\interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Kira\AppData\Local\Temp\ipykernel_5716\959565569.py", line 23, in <module>
    dataset = Dataset.from_pandas(df['Translation'])
  File "c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\datasets\arrow_dataset.py", line 838, in from_pandas
    table = InMemoryTable.from_pandas(
  File "c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\datasets\table.py", line 719, in from_pandas
    return cls(pa.Table.from_pandas(*args, **kwargs))
  File "pyarrow\\table.pxi", line 4623, in pyarrow.lib.Table.from_pandas
  File "c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\pyarrow\pandas_compat.py", line 572, in dataframe_to_arrays
    convert_fields) = _get_columns_to_convert(df, schema, preserve_index,
  File "c:\Users\Kira\anaconda3\envs\block_b\lib\site-pack

In [3]:
model_path = r"models\emotion_7"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
# Move model to device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
df['Translation'] = df['Translation'].apply(optimized_preprocessor)
df['Translation'] = df['Translation'].apply(remove_placeholders)
# Load or define df_test here
dataset = Dataset.from_pandas(df[['Translation']])

# Tokenize
def tokenize_function(example):
    return tokenizer(example['Translation'], padding="max_length", truncation=True, max_length=128)

tokenized_dataset_test = dataset.map(tokenize_function, batched=True)
tokenized_dataset_test = tokenized_dataset_test.remove_columns(["Translation"])
tokenized_dataset_test.set_format(type='torch', columns=['input_ids', 'attention_mask'])
# Recreate the trainer
trainer = Trainer(
    model=model,
    tokenizer=tokenizer
)
predictions_output = trainer.predict(tokenized_dataset_test)

# Extract logits and labels
logits = predictions_output.predictions

# Convert logits to predicted label indices
predicted_labels = logits.argmax(axis=1)

# Define label mappings happiness - sadness - anger - surprise - fear - disgust
label_mapping = {
    'disgust': 0,
    'happiness': 1,
    'anger': 2,
    'neutral': 3,
    'sadness': 4,
    'fear': 5,
    'surprise': 6
}
# Reverse mapping to decode label integers
reverse_label_mapping = {v: k for k, v in label_mapping.items()}

df['Core Emotion'] = predicted_labels
df['Core Emotion'] = df['Core Emotion'].map(reverse_label_mapping)
df

Map: 100%|██████████| 145/145 [00:00<00:00, 7393.67 examples/s]
C:\Users\Kira\AppData\Local\Temp\ipykernel_24676\3346119697.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\Kira\anaconda3\envs\block_b\lib\site-packages\transformers\models\roberta\modeling_roberta.py:370: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
100%|██████████| 19/19 [00:00<00:00, 29.01it/s]


,Sentence,Translation,Core Emotion
0,După un 1-0 tras muncit de ambele echipe și de...,after a [num]-[num] pulled worked by both team...,surprise
1,"Lea și Izabela pentru ei, Ionela și Diana pent...","lea and izabela for them, ionela and diana for...",neutral
2,"Încă o dată, ce este în cufăr, ce e în cutie, ...","once again, what is in the chest, what is in t...",surprise
3,Atenție!,attention!,neutral
4,"Ladies, pregătiți-vă!","ladies, get ready!",fear
...,...,...,...
140,Exilul a adus punct un duel copleșitor.,exile brought the point of an overwhelming duel,happiness
141,Ne pare nespus de rău pentru colegiile noastre...,we are so sorry for our dragon colleagues,sadness
142,Însă una e Lea și una e Izabela.,but one is lea and one is izabela,neutral
143,Asta e nebunia noastră!,this is our insanity!,anger


In [11]:
!pip install transformers[torch]

   ---------------------------------------- 0.0/330.9 kB ? eta -:--:--
   --- ------------------------------------ 30.7/330.9 kB ? eta -:--:--
   ---------------------------------------- 330.9/330.9 kB 5.1 MB/s eta 0:00:00


In [2]:
df = pd.read_csv("translated_data.csv", encoding='utf-8-sig')

In [6]:
df

,Sentence,Translation
0,După un 1-0 tras muncit de ambele echipe și de...,After a 1-0 pulled worked by both teams and Dr...
1,"Lea și Izabela pentru ei, Ionela și Diana pent...","Lea and Izabela for them, Ionela and Diana for..."
2,"Încă o dată, ce este în cufăr, ce e în cutie, ...","Once again, what's in the chest, what's in the..."
3,Atenție!,Attention!
4,"Ladies, pregătiți-vă!","Ladies, get ready!"
...,...,...
140,Exilul a adus punct un duel copleșitor.,Exile brought the point of an overwhelming duel.
141,Ne pare nespus de rău pentru colegiile noastre...,We're so sorry for our dragon colleagues.
142,Însă una e Lea și una e Izabela.,But one is Lea and one is Izabela.
143,Asta e nebunia noastră!,This is our insanity!


In [8]:
device

device(type='cpu')

In [1]:
import torch

# Check PyTorch version
print(f"PyTorch version: {torch.__version__}")

# Check if CUDA is available
if torch.cuda.is_available():
    print(f"CUDA is available. Version: {torch.version.cuda}")
else:
    print("CUDA is not available.")

PyTorch version: 2.4.1+cu124
CUDA is available. Version: 12.4


In [2]:
!nvidia-smi

Tue Apr  8 10:04:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.83                 Driver Version: 572.83         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   57C    P5             13W /  130W |     777MiB /   6144MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
     ---------------- ----------------------- 1.7/4.1 MB 36.2 MB/s eta 0:00:01
     -------------------------- ------------- 2.7/4.1 MB 29.1 MB/s eta 0:00:01
     -------------------------------- ------- 3.3/4.1 MB 26.5 MB/s eta 0:00:01
     ---------------------------------------- 4.1/4.1 MB 21.9 MB/s eta 0:00:00
     ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 GB 46.9 MB/s eta 0:00:54
     ---------------------------------------- 0.0/2.5 GB 35.7 MB/s eta 0:01:11
     ---------------------------------------- 0.0/2.5 GB 41.9 MB/s eta 0:01:00
     ---------------------------------------- 0.0/2.5 GB 41.3 MB/s eta 0:01:01
     ---------------------------------------- 0.0/2.5 GB 40.2 MB/s eta 0:01:03
     ---------------------------------------- 0.0/2.5 GB 38.6 MB/s eta 0:01:05
     ----

  You can safely remove it manually.
